# TEST-04: Persistent Long Signal (Re-Entry Audit)

**WP-5a Pipeline Audit** | 2026-04-16

**Intent:** Determine whether a signal that stays +1 for 100 consecutive bars generates one trade or many trades.

**Pre-registration:** If the backtester re-enters per bar on a persistent signal, this would explain the very high trade counts seen in some hypotheses. If it correctly treats persistent +1 as one position, the signal must flip to cause a new entry.

**Expected:**
- Signal: +1 for bars 0–99, then 0 forever
- Exactly 1 round-trip trade (entry at bar 1's open, exit at bar 100's open)
- Fees paid for 2 transactions only (entry + exit)

In [1]:
import sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, 'C:/GitHub/crypto-perp-wedge')
DATA_ROOT = Path('C:/GitHub/crypto-perp-wedge/data/processed')
GAPS_PATH = Path('C:/GitHub/crypto-perp-wedge/data/irreparable_gaps.json')

from lib.backtest import run_backtest

OHLCV_PATH = DATA_ROOT / 'binance_um_perp_BTCUSDT_4h_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet'
COSTS = {'fee_bps_taker': 4.0, 'fee_bps_maker': 2.0, 'slippage_bps': 2.0, 'initial_equity': 10_000.0}

def load_ohlcv(path):
    df = pd.read_parquet(path).copy()
    df.index = pd.to_datetime(df['close_time_ms'].astype('int64'), unit='ms', utc=True)
    return df[['open','high','low','close','volume']].astype(float).sort_index()

ohlcv = load_ohlcv(OHLCV_PATH)
n = len(ohlcv)
funding = pd.DataFrame({'funding_rate': []}, index=pd.DatetimeIndex([], tz='UTC'))

# Signal: +1 for first 100 bars, then 0
persistent_vals = [1.0]*100 + [0.0]*(n-100)
sig = pd.Series(persistent_vals, index=ohlcv.index, name='signal')

result = run_backtest(sig, ohlcv, funding, **COSTS, execution='next_bar_open',
                      stream='klines', interval='4h', gaps_path=GAPS_PATH)
print(f'Signal[0..4]:   {list(sig.iloc[:5].values)} (first 5 bars = +1)')
print(f'Signal[99..104]: {list(sig.iloc[99:105].values)} (transition from +1 to 0)')
print(f'Trade count:    {len(result.trades_df)} (expected: 1)')
print(f'Fees paid:      {result.fees_paid:.4f}')

Signal[0..4]:   [1.0, 1.0, 1.0, 1.0, 1.0] (first 5 bars = +1)
Signal[99..104]: [1.0, 0.0, 0.0, 0.0, 0.0] (transition from +1 to 0)
Trade count:    1 (expected: 1)
Fees paid:      6.4428


In [2]:
print('=== Trade details ===')
for _, row in result.trades_df.iterrows():
    print(f'  Direction:   {row["direction"]} (1 = long)')
    print(f'  Entry bar:   {row["entry_bar"]}')
    print(f'  Exit bar:    {row["exit_bar"]}')
    print(f'  Entry price: {row["entry_price"]:.2f}')
    print(f'  Exit price:  {row["exit_price"]:.2f}')
    print(f'  Gross PnL:   {row["gross_pnl"]:.2f}')
    print(f'  Net PnL:     {row["net_pnl"]:.2f}')
print()
print(f'Expected entry bar: {ohlcv.index[1]} (open of bar 1, signal at bar 0 was +1)')
print(f'Expected exit bar:  {ohlcv.index[100]} (open of bar 100, signal at bar 99 changed to 0)')

=== Trade details ===
  Direction:   1 (1 = long)
  Entry bar:   2020-01-01 07:59:59.999000+00:00
  Exit bar:    2020-01-17 23:59:59.999000+00:00
  Entry price: 7221.80
  Exit price:  8885.08
  Gross PnL:   1663.28
  Net PnL:     1650.40

Expected entry bar: 2020-01-01 07:59:59.999000+00:00 (open of bar 1, signal at bar 0 was +1)
Expected exit bar:  2020-01-17 23:59:59.999000+00:00 (open of bar 100, signal at bar 99 changed to 0)


In [3]:
trade_count_ok = len(result.trades_df) == 1
entry_bar_ok = result.trades_df.iloc[0]['entry_bar'] == ohlcv.index[1]
exit_bar_ok = result.trades_df.iloc[0]['exit_bar'] == ohlcv.index[100]
verdict = 'PASS' if (trade_count_ok and entry_bar_ok and exit_bar_ok) else 'FAIL'

print(f'=== VERDICT: {verdict} ===')
print(f'Trade count = 1:       {"PASS" if trade_count_ok else "FAIL"}')
print(f'Entry bar correct:     {"PASS" if entry_bar_ok else "FAIL"}')
print(f'Exit bar correct:      {"PASS" if exit_bar_ok else "FAIL"}')
print()
print('Conclusion: The backtester correctly treats a persistent +1 as ONE open position.')
print('No per-bar re-entry occurs. High trade counts in H-005 (934), H-006 (~many) are')
print('driven by genuine signal transitions in the strategy logic, not a re-entry bug.')

=== VERDICT: PASS ===
Trade count = 1:       PASS
Entry bar correct:     PASS
Exit bar correct:      PASS

Conclusion: The backtester correctly treats a persistent +1 as ONE open position.
No per-bar re-entry occurs. High trade counts in H-005 (934), H-006 (~many) are
driven by genuine signal transitions in the strategy logic, not a re-entry bug.
